## Dataset analysis: `student_attendance_list16.csv`

Purpose: daily attendance events (List16). Used to derive attendance trends per semester for CGPA prediction.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "student_attendance_list16.csv")
df.shape

(2126474, 5)

In [2]:
df.head()

,REG_NO,ACC_NO,DATE,SEMESTER_INDEX,STATUS
0,KM24B80/062,B27215,2024-05-10,1,PRESENT
1,KM24B80/062,B27215,2024-05-13,1,PRESENT
2,KM24B80/062,B27215,2024-05-14,1,PRESENT
3,KM24B80/062,B27215,2024-05-15,1,PRESENT
4,KM24B80/062,B27215,2024-05-16,1,PRESENT


In [3]:
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
df["STATUS"] = df["STATUS"].astype(str).str.upper().str.strip().replace({"P":"PRESENT","A":"ABSENT","L":"LATE"})

df.isna().mean().sort_values(ascending=False)

REG_NO            0.0
ACC_NO            0.0
DATE              0.0
SEMESTER_INDEX    0.0
STATUS            0.0
dtype: float64

In [4]:
df["STATUS"].value_counts(dropna=False)

STATUS
PRESENT    2020547
LATE         63376
ABSENT       42551
Name: count, dtype: int64

In [5]:
# Semestral attendance rates
g = df.dropna(subset=["REG_NO","SEMESTER_INDEX","DATE"]).groupby(["REG_NO","SEMESTER_INDEX"], as_index=False)
out = g.agg(attendance_days=("DATE","nunique"), present=("STATUS", lambda s: (s=="PRESENT").sum()), late=("STATUS", lambda s: (s=="LATE").sum()), absent=("STATUS", lambda s: (s=="ABSENT").sum()))
out["present_rate"] = out["present"] / out["attendance_days"].replace({0: np.nan})
out[["attendance_days","present_rate"]].describe().T

,count,mean,std,min,25%,50%,75%,max
attendance_days,27726.0,76.696025,12.518279,12.000000,79.000000,79.00,80.000000,80.0
present_rate,27726.0,0.950247,0.026559,0.666667,0.936709,0.95,0.974684,1.0


## Advanced analytics

Focus: attendance trend features by semester and association with CGPA (joined to transcript).

In [ ]:
import numpy as np
from analysis_utils import basic_profile, missingness_report, merge_to_transcript_for_cgpa

print(basic_profile(df))
missingness_report(df, top_n=20)

BasicProfile(rows=2126474, cols=5, dup_rows=0, null_cells=0)


,dtype,missing_rate,missing_count,nunique
REG_NO,str,0.0,0,5000
ACC_NO,str,0.0,0,5000
DATE,datetime64[us],0.0,0,1236
SEMESTER_INDEX,int64,0.0,0,10
STATUS,str,0.0,0,3


In [ ]:
trans = pd.read_csv(DATA_DIR / "student_transcript_list16.csv")
trans["CGPA"] = pd.to_numeric(trans["CGPA"], errors="coerce")

x = df.copy()
x["DATE"] = pd.to_datetime(x["DATE"], errors="coerce")
x["STATUS"] = x["STATUS"].astype(str).str.upper().str.strip().replace({"P":"PRESENT","A":"ABSENT","L":"LATE"})

x = x.dropna(subset=["REG_NO","SEMESTER_INDEX","DATE"])

g = x.groupby(["REG_NO","SEMESTER_INDEX"], as_index=False)
att_sem = g.agg(
    attendance_days=("DATE","nunique"),
    present_rate=("STATUS", lambda s: float((s=="PRESENT").mean())),
    late_rate=("STATUS", lambda s: float((s=="LATE").mean())),
    absent_rate=("STATUS", lambda s: float((s=="ABSENT").mean())),
    attendance_span_days=("DATE", lambda s: (s.max() - s.min()).days + 1),
)

joined = merge_to_transcript_for_cgpa(att_sem, trans, on=["REG_NO","SEMESTER_INDEX"], how="inner")
joined[["CGPA","present_rate","late_rate","absent_rate","attendance_days"]].corr(numeric_only=True)

,CGPA,present_rate,late_rate,absent_rate,attendance_days
CGPA,1.000000,-0.002423,0.001771,0.001621,0.011941
present_rate,-0.002423,1.000000,-0.767346,-0.624259,-0.013887
late_rate,0.001771,-0.767346,1.000000,-0.021920,0.011532
absent_rate,0.001621,-0.624259,-0.021920,1.000000,0.007602
attendance_days,0.011941,-0.013887,0.011532,0.007602,1.000000
